# Short Anchor Schema Variants for GLiNER2

This notebook tests short, natural-language label anchors for zero-shot Banking77 intent classification. It does not use Banking77 train examples, train labels, external LLMs, fine-tuning, LoRA, paid APIs, or GLiNER2 internals.


In [ ]:
PROJECT_NAME = "gliner2_short_anchor_schema"
MODEL_NAME = "fastino/gliner2-base-v1"
DATASET_NAME = "mteb/banking77"
MODE = "small"  # smoke / small / full
SEED = 42
SMOKE_N_EXAMPLES = 20
SMALL_PER_LABEL = 5
FULL_USE_ALL_TEST = True
CONFIRM_FULL_RUN = False
FORCE_RERUN = False
OUTPUT_DIR = "/content/gliner2_schema_outputs"
SHORT_ANCHOR_OUTPUT_SUBDIR = "short_anchor_schema"

RUN_SHORT_ANCHOR = True
RUN_QUERY_ABOUT_SHORT_ANCHOR = True
RUN_CUSTOMER_REQUEST_SHORT_ANCHOR = True
RUN_ISSUE_SHORT_ANCHOR = True
RUN_MINIMAL_CONTRASTIVE_ANCHOR = True
RUN_QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR = True
RUN_ORACLE_ANALYSIS = True

if MODE == "full" and not CONFIRM_FULL_RUN:
    raise RuntimeError(
        "MODE='full' requires CONFIRM_FULL_RUN=True. This prevents accidental full evaluation."
    )


## Install/import


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

OUTPUT_PATH = Path(OUTPUT_DIR) / SHORT_ANCHOR_OUTPUT_SUBDIR
PREDICTIONS_DIR = OUTPUT_PATH / "predictions"
TABLES_DIR = OUTPUT_PATH / "tables"
FIGURES_DIR = OUTPUT_PATH / "figures"
for path in [OUTPUT_PATH, PREDICTIONS_DIR, TABLES_DIR, FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)


def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content"),
        Path("/content/GLiNER2-demo"),
        Path("/content/gliner2_schema_wording"),
        Path("/content/drive/MyDrive/GLiNER2-demo"),
        Path("/content/drive/MyDrive/gliner2_schema_wording"),
    ]
    for root in candidates:
        if (root / "requirements-colab.txt").exists() and (root / "src").exists():
            return root.resolve()
    for pattern in ["*/requirements-colab.txt", "*/*/requirements-colab.txt"]:
        for req_path in Path("/content").glob(pattern):
            root = req_path.parent
            if (root / "src").exists():
                return root.resolve()
    return None


PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    repo_url = "https://github.com/daidai-su/GLiNER2-demo.git"
    clone_dir = Path("/content/GLiNER2-demo")
    print("Project files were not found in the runtime.")
    print(f"Trying to clone project files from {repo_url} ...")
    try:
        if not clone_dir.exists():
            subprocess.check_call(["git", "clone", "--depth", "1", repo_url, str(clone_dir)])
        PROJECT_ROOT = find_project_root()
    except Exception as exc:
        print(f"Automatic clone failed: {exc}")

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find project root. Upload or clone the full repository into Colab.")

print(f"Project root: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
)


## Environment check


In [ ]:
import hashlib
import random
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

from gliner2_project.analysis import (
    compare_method_examples,
    confusion_pairs_frame,
    method_metrics_frame,
    per_label_delta_frame,
    per_label_metrics_frame,
)
from gliner2_project.anchor_coverage import anchor_coverage_frame, anchor_table_frame, validate_anchor_coverage
from gliner2_project.data_utils import ensure_label_text_column, stratified_subset, unique_label_texts
from gliner2_project.ensemble import MEAN_CONFIDENCE_ENSEMBLE, mean_confidence_ensemble
from gliner2_project.env_utils import print_environment_info
from gliner2_project.gliner2_wrappers import load_gliner2_model, predict_one_classification
from gliner2_project.plotting import plot_metric_bar, plot_top_label_improvements
from gliner2_project.schema_variants import (
    BANKING_REQUEST_LABEL,
    CUSTOMER_INTENT_LABEL,
    ENSEMBLE_INPUT_METHODS,
    PLAIN_LABEL,
    QUERY_ABOUT_LABEL,
    build_schema_variant,
)
from gliner2_project.short_anchor_schema import (
    CUSTOMER_REQUEST_SHORT_ANCHOR,
    ISSUE_SHORT_ANCHOR,
    MINIMAL_CONTRASTIVE_ANCHOR,
    QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR,
    QUERY_ABOUT_SHORT_ANCHOR,
    SHORT_ANCHOR,
    build_short_anchor_schema,
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
RUN_START_UTC = datetime.now(timezone.utc).isoformat()
environment_info = print_environment_info()
print(f"Run started at: {RUN_START_UTC}")


## Load GLiNER2


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    print("WARNING: GPU is unavailable. GLiNER2 inference may be slow.")
else:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")

model = load_gliner2_model(MODEL_NAME, device=DEVICE)
print("Model loaded.")


## Load Banking77 test split only


In [ ]:
test_split = ensure_label_text_column(load_dataset(DATASET_NAME, split="test"))
test_examples = []
for idx, row in enumerate(test_split):
    item = dict(row)
    item.setdefault("example_id", idx)
    item["text"] = str(item["text"])
    item["label_text"] = str(item["label_text"])
    test_examples.append(item)

label_texts = unique_label_texts(test_examples)
print(f"Test examples: {len(test_examples)}")
print(f"Labels: {len(label_texts)}")
print("No Banking77 train split is loaded in this notebook.")


## Build evaluation subset


In [ ]:
if MODE == "smoke":
    eval_examples = list(test_examples[:SMOKE_N_EXAMPLES])
elif MODE == "small":
    eval_examples = stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
elif MODE == "full":
    eval_examples = list(test_examples) if FULL_USE_ALL_TEST else stratified_subset(test_examples, per_label=SMALL_PER_LABEL, seed=SEED)
else:
    raise ValueError(f"Unknown MODE: {MODE}")

example_ids = [example["example_id"] for example in eval_examples]
print(f"Mode: {MODE}")
print(f"Evaluated examples: {len(eval_examples)}")


## Build and validate short anchors


In [ ]:
validate_anchor_coverage(label_texts)
anchor_table = anchor_table_frame(label_texts)
coverage = anchor_coverage_frame(label_texts)
anchor_table.to_csv(TABLES_DIR / "short_anchor_table.csv", index=False)
coverage.to_csv(TABLES_DIR / "short_anchor_coverage.csv", index=False)
display(coverage)
display(anchor_table.head(30))


## Cache helpers


In [ ]:
def json_safe(value):
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def load_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def save_jsonl(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=json_safe) + "\n")


BASE_MANIFEST = {
    "project_name": PROJECT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": "test",
    "mode": MODE,
    "num_examples": len(eval_examples),
    "example_ids": example_ids,
    "seed": SEED,
}


def manifest_for(method_name, extra=None):
    manifest = dict(BASE_MANIFEST)
    manifest["method"] = method_name
    if extra:
        manifest.update(extra)
    manifest["manifest_hash"] = stable_hash({k: v for k, v in manifest.items() if k != "manifest_hash"})
    return manifest


def cache_manifest_matches(path, expected):
    path = Path(path)
    if not path.exists():
        return False
    try:
        cached = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False
    return cached == expected


def load_method_cache(method_name, manifest):
    if FORCE_RERUN:
        return None
    pred_path = PREDICTIONS_DIR / f"{method_name}.jsonl"
    manifest_path = PREDICTIONS_DIR / f"{method_name}.manifest.json"
    if pred_path.exists() and cache_manifest_matches(manifest_path, manifest):
        print(f"Loading cached {method_name}")
        return load_jsonl(pred_path)
    return None


def save_method_cache(method_name, rows, manifest):
    save_jsonl(rows, PREDICTIONS_DIR / f"{method_name}.jsonl")
    (PREDICTIONS_DIR / f"{method_name}.manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
        encoding="utf-8",
    )


## Run baseline and short-anchor methods


In [ ]:
predictions_by_method = {}


def run_candidate_schema(method_name, candidate_labels, candidate_to_canonical, extra_manifest=None):
    manifest = manifest_for(method_name, extra_manifest)
    cached = load_method_cache(method_name, manifest)
    if cached is not None:
        predictions_by_method[method_name] = cached
        return cached
    rows = []
    for example in tqdm(eval_examples, desc=method_name):
        rows.append(
            predict_one_classification(
                model=model,
                text=example["text"],
                candidate_labels=list(candidate_labels),
                candidate_to_canonical=dict(candidate_to_canonical),
                method_name=method_name,
                example_id=example["example_id"],
                gold_label=example["label_text"],
            )
        )
    save_method_cache(method_name, rows, manifest)
    predictions_by_method[method_name] = rows
    return rows


def add_effective_latency_from_components(rows, component_methods):
    indexes = {
        method: {row.get("example_id"): row for row in predictions_by_method.get(method, [])}
        for method in component_methods
    }
    for row in rows:
        latencies = [
            indexes[method].get(row.get("example_id"), {}).get("latency_sec")
            for method in component_methods
        ]
        if all(value is not None for value in latencies):
            row["latency_sec"] = float(sum(float(value) for value in latencies))
    return rows


for method_name in ENSEMBLE_INPUT_METHODS:
    candidates, mapping = build_schema_variant(label_texts, method_name)
    run_candidate_schema(
        method_name,
        candidates,
        mapping,
        extra_manifest={"schema_type": "deterministic_baseline"},
    )

mean_rows = mean_confidence_ensemble(predictions_by_method)
mean_rows = add_effective_latency_from_components(mean_rows, ENSEMBLE_INPUT_METHODS)
for row in mean_rows:
    row["method"] = MEAN_CONFIDENCE_ENSEMBLE
predictions_by_method[MEAN_CONFIDENCE_ENSEMBLE] = mean_rows
save_jsonl(mean_rows, PREDICTIONS_DIR / f"{MEAN_CONFIDENCE_ENSEMBLE}.jsonl")

short_methods = []
if RUN_SHORT_ANCHOR:
    short_methods.append(SHORT_ANCHOR)
if RUN_QUERY_ABOUT_SHORT_ANCHOR:
    short_methods.append(QUERY_ABOUT_SHORT_ANCHOR)
if RUN_CUSTOMER_REQUEST_SHORT_ANCHOR:
    short_methods.append(CUSTOMER_REQUEST_SHORT_ANCHOR)
if RUN_ISSUE_SHORT_ANCHOR:
    short_methods.append(ISSUE_SHORT_ANCHOR)
if RUN_MINIMAL_CONTRASTIVE_ANCHOR:
    short_methods.append(MINIMAL_CONTRASTIVE_ANCHOR)
if RUN_QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR:
    short_methods.append(QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR)

for method_name in short_methods:
    candidates, mapping = build_short_anchor_schema(label_texts, method_name)
    run_candidate_schema(
        method_name,
        candidates,
        mapping,
        extra_manifest={"schema_type": "short_anchor"},
    )


## Analysis


In [ ]:
def method_index(method):
    return {row.get("example_id"): row for row in predictions_by_method.get(method, [])}


def paired_comparison(baseline, comparison):
    base = method_index(baseline)
    comp = method_index(comparison)
    rows = []
    for eid, base_row in base.items():
        if eid not in comp:
            continue
        gold = base_row.get("gold_label")
        base_pred = base_row.get("predicted_canonical")
        comp_pred = comp[eid].get("predicted_canonical")
        rows.append(
            {
                "example_id": eid,
                "text": base_row.get("text"),
                "gold_label": gold,
                "baseline_method": baseline,
                "comparison_method": comparison,
                "baseline_prediction": base_pred,
                "comparison_prediction": comp_pred,
                "baseline_correct": base_pred == gold,
                "comparison_correct": comp_pred == gold,
            }
        )
    frame = pd.DataFrame(rows)
    if frame.empty:
        return pd.DataFrame(), pd.DataFrame()
    summary = pd.DataFrame(
        [
            {
                "baseline_method": baseline,
                "comparison_method": comparison,
                "num_examples": int(len(frame)),
                "both_correct": int((frame["baseline_correct"] & frame["comparison_correct"]).sum()),
                "both_wrong": int((~frame["baseline_correct"] & ~frame["comparison_correct"]).sum()),
                "baseline_correct_comparison_wrong": int((frame["baseline_correct"] & ~frame["comparison_correct"]).sum()),
                "baseline_wrong_comparison_correct": int((~frame["baseline_correct"] & frame["comparison_correct"]).sum()),
                "net_gain": int((~frame["baseline_correct"] & frame["comparison_correct"]).sum())
                - int((frame["baseline_correct"] & ~frame["comparison_correct"]).sum()),
                "accuracy_delta": float(frame["comparison_correct"].mean() - frame["baseline_correct"].mean()),
            }
        ]
    )
    return summary, frame


def oracle_analysis(methods):
    available = [method for method in methods if method in predictions_by_method]
    if not available:
        return pd.DataFrame(), pd.DataFrame()
    indexes = {method: method_index(method) for method in available}
    first = available[0]
    rows = []
    for eid, base_row in indexes[first].items():
        gold = base_row.get("gold_label")
        method_predictions = {
            method: indexes[method].get(eid, {}).get("predicted_canonical")
            for method in available
        }
        correct_methods = [method for method, pred in method_predictions.items() if pred == gold]
        vote_counts = Counter(pred for pred in method_predictions.values() if pred is not None)
        rows.append(
            {
                "example_id": eid,
                "text": base_row.get("text"),
                "gold_label": gold,
                "oracle_correct": bool(correct_methods),
                "num_correct_variants": len(correct_methods),
                "correct_methods": correct_methods,
                "max_vote_count": max(vote_counts.values()) if vote_counts else 0,
                **method_predictions,
            }
        )
    frame = pd.DataFrame(rows)
    summary = pd.DataFrame(
        [
            {
                "methods": ", ".join(available),
                "num_examples": int(len(frame)),
                "oracle_accuracy": float(frame["oracle_correct"].mean()) if not frame.empty else 0.0,
                "oracle_correct": int(frame["oracle_correct"].sum()) if not frame.empty else 0,
            }
        ]
    )
    return summary, frame


## Save artifacts


In [ ]:
results_summary = method_metrics_frame(predictions_by_method)
results_summary.to_csv(TABLES_DIR / "short_anchor_results_summary.csv", index=False)
display(results_summary.sort_values("accuracy", ascending=False))

paired_summaries = []
paired_examples = []
for baseline, comparison in [
    (PLAIN_LABEL, QUERY_ABOUT_SHORT_ANCHOR),
    (QUERY_ABOUT_LABEL, QUERY_ABOUT_SHORT_ANCHOR),
    (PLAIN_LABEL, QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR),
    (QUERY_ABOUT_LABEL, QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR),
]:
    if baseline in predictions_by_method and comparison in predictions_by_method:
        summary, examples = paired_comparison(baseline, comparison)
        if not summary.empty:
            paired_summaries.append(summary)
        if not examples.empty:
            paired_examples.append(examples)
paired_summary = pd.concat(paired_summaries, ignore_index=True) if paired_summaries else pd.DataFrame()
paired_all_examples = pd.concat(paired_examples, ignore_index=True) if paired_examples else pd.DataFrame()
paired_summary.to_csv(TABLES_DIR / "paired_comparisons.csv", index=False)
paired_all_examples.to_csv(TABLES_DIR / "paired_comparison_examples.csv", index=False)
display(paired_summary)

oracle_methods = [
    PLAIN_LABEL,
    QUERY_ABOUT_LABEL,
    SHORT_ANCHOR,
    QUERY_ABOUT_SHORT_ANCHOR,
    CUSTOMER_REQUEST_SHORT_ANCHOR,
    MINIMAL_CONTRASTIVE_ANCHOR,
    QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR,
]
oracle_summary, oracle_examples = oracle_analysis(oracle_methods if RUN_ORACLE_ANALYSIS else [])
oracle_summary.to_csv(TABLES_DIR / "oracle_analysis.csv", index=False)
oracle_examples.to_csv(TABLES_DIR / "oracle_examples.csv", index=False)
display(oracle_summary)

if not oracle_examples.empty:
    correct_vote_dist = (
        oracle_examples["num_correct_variants"]
        .value_counts()
        .sort_index()
        .rename_axis("num_correct_variants")
        .reset_index(name="num_examples")
    )
    max_vote_dist = (
        oracle_examples["max_vote_count"]
        .value_counts()
        .sort_index()
        .rename_axis("max_vote_count")
        .reset_index(name="num_examples")
    )
else:
    correct_vote_dist = pd.DataFrame()
    max_vote_dist = pd.DataFrame()
correct_vote_dist.to_csv(TABLES_DIR / "correct_vote_count_distribution.csv", index=False)
max_vote_dist.to_csv(TABLES_DIR / "max_vote_count_distribution.csv", index=False)
display(correct_vote_dist)

if QUERY_ABOUT_LABEL in predictions_by_method:
    for comparison in [QUERY_ABOUT_SHORT_ANCHOR, QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR]:
        if comparison in predictions_by_method:
            improved, degraded = compare_method_examples(predictions_by_method, QUERY_ABOUT_LABEL, comparison)
            improved.to_csv(TABLES_DIR / f"improved_examples_{comparison}_vs_query_about.csv", index=False)
            degraded.to_csv(TABLES_DIR / f"degraded_examples_{comparison}_vs_query_about.csv", index=False)
            if comparison == QUERY_ABOUT_SHORT_ANCHOR:
                improved.to_csv(TABLES_DIR / "improved_examples_vs_query_about.csv", index=False)
                degraded.to_csv(TABLES_DIR / "degraded_examples_vs_query_about.csv", index=False)

per_label = per_label_metrics_frame(predictions_by_method)
per_label.to_csv(TABLES_DIR / "per_label_metrics.csv", index=False)
delta_frames = []
for comparison in [QUERY_ABOUT_SHORT_ANCHOR, QUERY_ABOUT_MINIMAL_CONTRASTIVE_ANCHOR]:
    if QUERY_ABOUT_LABEL in predictions_by_method and comparison in predictions_by_method:
        delta_frames.append(per_label_delta_frame(per_label, QUERY_ABOUT_LABEL, comparison))
per_label_delta = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
per_label_delta.to_csv(TABLES_DIR / "per_label_delta_vs_query_about.csv", index=False)

confusion_frames = []
for method, rows in predictions_by_method.items():
    pairs = confusion_pairs_frame(rows, top_n=50)
    if not pairs.empty:
        pairs["method"] = method
        confusion_frames.append(pairs)
confusions = pd.concat(confusion_frames, ignore_index=True) if confusion_frames else pd.DataFrame()
confusions.to_csv(TABLES_DIR / "confusion_pairs.csv", index=False)

if not results_summary.empty:
    plot_metric_bar(
        results_summary,
        "accuracy",
        FIGURES_DIR / "short_anchor_method_comparison_accuracy.png",
        "Short Anchor Method Accuracy",
    )
    plot_metric_bar(
        results_summary,
        "macro_f1",
        FIGURES_DIR / "short_anchor_method_comparison_macro_f1.png",
        "Short Anchor Method Macro F1",
    )
if not per_label_delta.empty:
    plot_top_label_improvements(
        per_label_delta,
        FIGURES_DIR / "top_label_improvements_short_anchor_vs_query_about.png",
    )


## Save manifest and final summary


In [ ]:
RUN_END_UTC = datetime.now(timezone.utc).isoformat()
manifest = {
    "project_name": PROJECT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "split": "test",
    "mode": MODE,
    "num_examples": len(eval_examples),
    "example_ids": example_ids,
    "seed": SEED,
    "device": DEVICE,
    "no_train_split_loaded": True,
    "methods": list(predictions_by_method),
    "start_time_utc": RUN_START_UTC,
    "end_time_utc": RUN_END_UTC,
    "environment": environment_info,
}
(OUTPUT_PATH / "short_anchor_run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=json_safe),
    encoding="utf-8",
)

print("=== Short anchor results ===")
display(results_summary.sort_values("accuracy", ascending=False))
print("\n=== Paired comparisons ===")
display(paired_summary)
print("\n=== Oracle analysis ===")
display(oracle_summary)
print("\n=== Correct vote count distribution ===")
display(correct_vote_dist)
print(f"Artifacts saved under: {OUTPUT_PATH}")
print("\nRecommendation rule:")
print("- If no method beats query_about_label by at least +0.02 accuracy on small, do not recommend full run.")
print("- If a method improves substantially, run full only once with that selected method.")
